In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data 

In [2]:
df = pd.read_csv('../data/consumer_complaints.csv')

/tmp/ipykernel_453131/484999383.py:1: DtypeWarning: Columns (5,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/consumer_complaints.csv')


In [3]:
print(f"Full dataset shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Full dataset shape: (555957, 18)

Columns:
['date_received', 'product', 'sub_product', 'issue', 'sub_issue', 'consumer_complaint_narrative', 'company_public_response', 'company', 'state', 'zipcode', 'tags', 'consumer_consent_provided', 'submitted_via', 'date_sent_to_company', 'company_response_to_consumer', 'timely_response', 'consumer_disputed?', 'complaint_id']

Missing values:
date_received                        0
product                              0
sub_product                     158322
issue                                0
sub_issue                       343335
consumer_complaint_narrative    489151
company_public_response         470833
company                              0
state                             4887
zipcode                           4505
tags                            477998
consumer_consent_provided       432499
submitted_via                        0
date_sent_to_company                 0
company_response_to_consumer         0
timely_response                 

# Filter rows with narrative text only

In [5]:
df_with_text = df[df['consumer_complaint_narrative'].notna()].copy()

In [6]:
print(f"Rows with narrative: {df_with_text.shape[0]}")

Rows with narrative: 66806


In [7]:
print("Product categories:")
print(df_with_text['product'].value_counts().head(10))

Product categories:
product
Debt collection            17552
Mortgage                   14919
Credit reporting           12526
Credit card                 7929
Bank account or service     5711
Consumer Loan               3678
Student loan                2128
Prepaid card                 861
Payday loan                  726
Money transfers              666
Name: count, dtype: int64


In [9]:
print(f"Issue Types: ")
print(df_with_text['issue'].value_counts().head(10))

Issue Types: 
issue
Incorrect information on credit report      8352
Cont'd attempts collect debt not owed       7520
Loan servicing, payments, escrow account    6554
Loan modification,collection,foreclosure    5163
Disclosure verification of debt             2912
Communication tactics                       2759
Account opening, closing, or management     2228
Credit reporting company's investigation    1990
Managing the loan or lease                  1650
False statements or representation          1647
Name: count, dtype: int64


In [11]:
# Sample narratives
print("Sample complaint narratives:")
for text in df_with_text['consumer_complaint_narrative'].head(3):
    print(f"\n{text[:300]}")
    print("---")

Sample complaint narratives:

XXXX has claimed I owe them {$27.00} for XXXX years despite the PROOF of PAYMENT I sent them : canceled check and their ownPAID INVOICE for {$27.00}! 
They continue to insist I owe them and collection agencies are after me. 
How can I stop this harassment for a bill I already paid four years ago? 

---

Due to inconsistencies in the amount owed that I was told by M & T Bank and the amount that was reported to the credit reporting agencies, I was advised to write a good will letter in order to address the issue and request the negative entry be removed from my credit report all together. I had a veh
---

In XX/XX/XXXX my wages that I earned at my job decreased by almost half, by XX/XX/XXXX I knew I was in trouble with my home loan. I began contacting WFB whom my home loan is with, for assitance and options. 
In early XX/XX/XXXX I began the Loan Modification process with Wells Fargo Bank. I was told 
---


# Sample 25,000 rows

In [12]:
df_sample = df_with_text.sample(25000, random_state=42).reset_index(drop=True)
print(f"Final dataset shape: {df_sample.shape}")
print(f"\nColumns we'll use:")
print(df_sample[['consumer_complaint_narrative', 'product', 'issue']].head(3))

Final dataset shape: (25000, 18)

Columns we'll use:
                        consumer_complaint_narrative          product  \
0  I have been battling with portfolio recovery a...  Debt collection   
1  In the fall of XXXX I applied for a mortgage m...         Mortgage   
2  i tried to call this number and i can not call...      Credit card   

                                      issue  
0     Cont'd attempts collect debt not owed  
1  Loan servicing, payments, escrow account  
2                          Billing disputes  



# Text cleaning

In [14]:
import re

In [16]:
def clean_narrative(text):
    text = re.sub(r'XXXX','REDACTED',text) # Remove XXXX redactions — replace with generic token

    text = re.sub(r'\{\$[\d,.]+\}','AMOUNT',text) # Remove dollar amounts in curly braces {$27.00}

    text = re.sub(r'\s+', ' ',text ).strip() # Remove extra whitespace

    return text[:512]

In [17]:
df_sample['clean_narrative'] = df_sample['consumer_complaint_narrative'].apply(clean_narrative)

In [18]:
print("Before cleaning:")
print(df_sample['consumer_complaint_narrative'].iloc[0][:200])
print("\nAfter cleaning:")
print(df_sample['clean_narrative'].iloc[0][:200])

Before cleaning:
I have been battling with portfolio recovery and Foster, Garbus & Garbus for over a year regarding a debt that is not mine. I continue receiving letters from Foster, Garbus XXXX Garbus regarding same 

After cleaning:
I have been battling with portfolio recovery and Foster, Garbus & Garbus for over a year regarding a debt that is not mine. I continue receiving letters from Foster, Garbus REDACTED Garbus regarding s


In [19]:
df_sample[['complaint_id', 'clean_narrative', 'product', 'issue']].to_csv(
    '../data/complaints_clean.csv', index=False
)
print(f"Saved {len(df_sample)} cleaned complaints")
print(f"\nProduct distribution:")
print(df_sample['product'].value_counts())

Saved 25000 cleaned complaints

Product distribution:
product
Debt collection            6522
Mortgage                   5562
Credit reporting           4638
Credit card                3077
Bank account or service    2184
Consumer Loan              1348
Student loan                789
Prepaid card                317
Payday loan                 277
Money transfers             246
Other financial service      40
Name: count, dtype: int64
